In [ ]:
### ----------- initialization ----------- ###
# packages loader + locale
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

use_package <- function(pkg) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
# packages
pkgs <- c("rio","tidyverse","glmnet","pls","caret","leaps","MASS","lars","AER","car","pROC","broom","boot","mgcv")
invisible(lapply(pkgs, use_package))

# seed for reproducibility 
set.seed(2025)


Exercise 1

In [ ]:
### ----------- Exercise 1: BODYFAT (PCR, PLS, subsets, LASSO) ----------- ###

# parameters
www <- "https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/bodyfat2.xlsx"
local_file <- "bodyfat2.xlsx"

# --------------------------- LOAD DATA ----------------------------------

# Download if file does not exits 
if (!file.exists(local_file)) {
  download.file(www, destfile = local_file, mode = "wb")
}

bodyfat <- rio::import(local_file)

# Print column names (helpful for debugging)
print(colnames(bodyfat))

# Remove missing rows
bodyfat <- dplyr::filter(bodyfat, complete.cases(bodyfat))

# ---------------------- SPLIT X / y -------------------------------------

# Response
y <- bodyfat$bodyfat

# Explanatory variables: remove bodyfat + density
X <- dplyr::select(bodyfat, -bodyfat, -density)

# Matrix for regression methods
Xmat <- stats::model.matrix(~ . - 1, data = X)

In [ ]:
# ---------------------- Covariance  -------------------------------------

# covariance matrix
cor_matrix <- cor(Xmat)
print('Covariance matrix')
cor_matrix

# eigenvalues of covariance matrix
eigs <- eigen(cor_matrix)$values
print('Eigenvalues')
eigs

# condition index (Belsley)
condition_index <- sqrt(max(eigs) / eigs)
condition_index

plot(condition_index, type = "b", main = "Condition Index", xlab='Eigenvalue idx', ylab='Belsley value of index')


We see that first 10 Belsley values of columns are below 10. That means that they contain only 1/10 of new information compared to highest eigenvalue. We set this as threshold so we can suspect ill-conditioning in next features. 

In [ ]:
# ------------------ PCR with LOOCV ------------------
print('--------- PCR with LOOCV ---------')
pcr_model <- pcr(
  y ~ Xmat,
  scale = TRUE,
  validation = "LOO"
)

summary(pcr_model) 

pcr_msep <- RMSEP(pcr_model)

# ------------------ PLS with LOOCV ------------------
print('--------- PLS with LOOCV ---------')
pls_model <- plsr(
  y ~ Xmat,
  scale = TRUE,
  validation = "LOO"
)

summary(pls_model)

pls_msep <- RMSEP(pls_model)

print('--------- Comparison plot ---------')
# Wyciągamy MSEP z obiektów pls::RMSEP
pcr_msep_values <- as.vector(pcr_msep$val[1,,])   # MSEP dla PCR
pls_msep_values <- as.vector(pls_msep$val[1,,])   # MSEP dla PLS

# Liczba komponentów (od 0)
components <- seq_along(pcr_msep_values) - 1

# ------------------ PLT ------------------
# colors
col_Y_pcr <- "blue"
col_X_pcr <- "lightblue"
col_Y_pls <- "red"
col_X_pls <- "pink"


# plot - LOOCV comparison
## plotting 
plot(
  components, pcr_msep_values,
  type = "b", pch = 19, col = col_Y_pcr,
  xlab = "Number of Components",
  ylab = "MSEP (LOOCV)",
  main = "PCR vs PLS – LOOCV MSEP Comparison"
)
lines(components, pls_msep_values, type = "b", pch = 17, col = col_Y_pls)


legend(
  "topright",
  legend = c("PCR", "PLS"),
  col = c(col_Y_pcr, col_Y_pls),
  pch = c(19, 17),
  lty = 1
)

We see from summaries and `PCR vs PLS - LOOCV MSEP Comparison` that PCR PLS is much better method, we can obtain best methods, already with 4/5 components where PCR obtains same results at 12 (from 13) components - we do not reduce any dimensions.

In [ ]:
# ---------------------- LASSO REGRESSION (LOOCV) ----------------------

# assigning values
X_lasso <- as.matrix(Xmat)
y_lasso <- y

# model LASSO (alpha = 1)
lasso_cv <- cv.glmnet(
  X_lasso,
  y_lasso,
  alpha = 1,              # LASSO
  nfolds = nrow(X_lasso), # LOOCV
  standardize = TRUE
)

# coefficens for lambda.min and lambda.1se
coef_min  <- coef(lasso_cv, s = "lambda.min")
coef_1se  <- coef(lasso_cv, s = "lambda.1se")

coef_min
coef_1se

# wykres ścieżek LASSO
plot(lasso_cv, main="Lasso_cv cross validation curve")
plot(lasso_cv$glmnet.fit, xvar = "lambda", label = TRUE, main="lasso_cv regularization path")

We will focus on `lasso_cv cross validation curve`(1) and `lasso_cv regularization path`(2) plots. 

The first plot(1) shows, how how MSE changes due to different $\lambda$ parameters.From (1) we can see that minimum is reached around $-\log(\lambda) \sim 4$ (first dotted vertical line). But more stable $\lambda$ parametr is choose on first dotted line 1-SE, were we get rid of most parameters and our model is more interpretative. 

The second plot shows how fast when variables enter the model, that first were variables with indices, and they impact the strongest.

Combining LASSO model under $\lambda$.1se seletcs only small subset of variables with comparable predictive accuracy, to the full model. WE can see that 6, 3, 1 ,13 were most important 



In [ ]:
### ----------- Exercise 2: Boston Housing (OLS, penalised methods) ----------- ###

In [ ]:
### ----------- Exercise 2: Boston Housing (OLS, penalised methods) ----------- ###
# parameters
## default local path (zmień jeśli masz inny)
local_boston <- "boston_corrected.txt"
url_boston <- "https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/boston_corrected.txt"

# problem solving
# load file (try local, otherwise download)
if (!file.exists(local_file)) {
  download.file(url_boston, destfile = local_boston, mode = "wb")
}
# read, removing header lines if present (note in PDF: first few lines are text)
raw <- readLines(local_boston)
# find first line that starts like numeric data (heuristic: line with numbers separated by spaces)
start_idx <- which(grepl("^\\s*\\d", raw))[1]
data_lines <- raw[start_idx:length(raw)]
tmpfile <- tempfile()
writeLines(data_lines, tmpfile)
boston <- read.table(tmpfile, header = FALSE)
# If file has extra columns, inspect
print(dim(boston))
head(boston)

# If you have a column names vector from course, set them:
colnames(boston) <- c("CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS","RAD","TAX","PTRATIO","B","LSTAT","MEDV")
# Response: log(MEDV)
boston$logMEDV <- log(boston$MEDV)

# remove incomplete instantiations explicitly (note: PDF warns some rows incomplete)
boston <- boston[complete.cases(boston), ]

# OLS
ols_mod <- lm(logMEDV ~ CRIM+ZN+INDUS+CHAS+NOX+RM+AGE+DIS+RAD+TAX+PTRATIO+B+LSTAT, data = boston)
summary(ols_mod)
# Diagnostic plots
par(mfrow = c(2,2))
plot(ols_mod)

# Penalised regression: glmnet (ridge, lasso, elastic net)
Xb <- as.matrix(boston %>% select(CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT))
yb <- boston$logMEDV

# Standardize is default in glmnet
cv_lasso_b <- cv.glmnet(Xb, yb, alpha = 1) # LASSO
cv_ridge_b <- cv.glmnet(Xb, yb, alpha = 0) # ridge
# Elastic net example (alpha = 0.5)
cv_enet_b <- cv.glmnet(Xb, yb, alpha = 0.5)

# Coefficients
coef(cv_lasso_b, s = "lambda.min")
coef(cv_ridge_b, s = "lambda.min")
coef(cv_enet_b, s = "lambda.min")

# Compare CV errors (default 10-fold)
c(lasso_min = min(cv_lasso_b$cvm), ridge_min = min(cv_ridge_b$cvm), enet_min = min(cv_enet_b$cvm))

# Compare with OLS via cross-validation (10-fold) using caret
library(caret)
ctrl <- trainControl(method = "cv", number = 10)
set.seed(2025)
train_ols <- train(logMEDV ~ CRIM+ZN+INDUS+CHAS+NOX+RM+AGE+DIS+RAD+TAX+PTRATIO+B+LSTAT,
                   data = boston, method = "lm", trControl = ctrl)
train_lasso <- train(x = Xb, y = yb, method = "glmnet", trControl = ctrl,
                     tuneGrid = expand.grid(alpha = 1, lambda = cv_lasso_b$lambda))
resamps <- resamples(list(OLS = train_ols, LASSO = train_lasso))
summary(resamps)

#### Metodologia

* **Cel**: dopasować model Poissona do liczby wizyt (`DoctorVisits` z pakietu `AER`) i sprawdzić poprawność modelu.
* **Wczytanie danych**:

  * `data("DoctorVisits", package = "AER")`. Sprawdzamy nazwę kolumny z licznikami.
* **Model podstawowy**:

  * `glm(..., family = poisson)` z log-linkiem.
* **Diagnostyka adequacy**:

  * **Overdispersion**: obliczamy `deviance/df.residual` (wartość istotnie >1 wskazuje overdispersion).
  * Porównujemy wariancję próby z wartością średnią (dla Poissona var=mean).
  * Sprawdzamy wielokrotne zero (zero-inflation).
* **Alternatywy gdy Poisson zawodny**:

  * `quasipoisson` (koryguje wariancję),
  * `MASS::glm.nb` (negative binomial) — modeluje dodatkowe składowe wariancji,
  * `pscl::zeroinfl` (zero-inflated Poisson/NegBin) jeśli duża liczba zer.
* **Diagnostyka**:

  * Wykresy reszt, residuals typu Pearson, miary wpływu (`influence.measures`).
* **Uwagi**:

  * Diagnostyka counts jest trudniejsza — wysoki RSS niekoniecznie oznacza zły model (patrz notka w zadaniu).
  * Jeśli overdispersion obecne, preferować NB lub quasi-Poisson; jeśli dużo zer i struktura dwufazowa — rozważyć ZIP/ZINB.


In [ ]:
### ----------- Exercise 3: Doctor Visits (AER) — Poisson / overdispersion diagnostics ----------- ###
# parameters
## load package AER
library(AER)
data("DoctorVisits", package = "AER")

# inspect
str(DoctorVisits)
summary(DoctorVisits)

# model: Poisson regression for number of visits (assuming column named 'visits' — adapt if different)
# check actual column names
names(DoctorVisits)
# Typical variables in this dataset: visits (count), income, health, etc. Adjust formula accordingly.
# Here use a generic formula: visits ~ income + health + age + gender (edit if needed)
# find a count column
count_col <- names(DoctorVisits)[sapply(DoctorVisits, is.numeric)][1] # heuristic
count_col

# For explicitness, assume 'visits' is named 'visits' in dataset; if not, replace with actual name
if ("visits" %in% names(DoctorVisits)) {
  formula_pois <- as.formula("visits ~ . - id") # remove id if exists
} else {
  # fallback: pick the first numeric column as response (user should adapt if different)
  response <- count_col
  predictors <- setdiff(names(DoctorVisits), response)
  formula_pois <- as.formula(paste(response, "~", paste(predictors, collapse = " + ")))
}
# Fit Poisson
pois_mod <- glm(formula_pois, family = poisson(link = "log"), data = DoctorVisits)
summary(pois_mod)

# Overdispersion test: compare residual deviance to df
dispersion_ratio <- deviance(pois_mod) / df.residual(pois_mod)
cat("Dispersion ratio (deviance/df):", dispersion_ratio, "\n")

# Check for zero-inflation: compare mean vs variance
count_vec <- DoctorVisits[[all.vars(formula_pois)[1]]]
cat("Mean:", mean(count_vec), "Var:", var(count_vec), "\n")
# If var >> mean => overdispersion

# Robust alternatives:
# 1) Quasi-Poisson
qpois_mod <- glm(formula_pois, family = quasipoisson(link = "log"), data = DoctorVisits)
summary(qpois_mod)

# 2) Negative binomial (MASS::glm.nb)
nb_mod <- try(MASS::glm.nb(formula_pois, data = DoctorVisits), silent = TRUE)
if (class(nb_mod) != "try-error") {
  summary(nb_mod)
}

# 3) Zero-inflated models (pscl package) -- optional
if (!requireNamespace("pscl", quietly = TRUE)) install.packages("pscl", repos = "https://cloud.r-project.org")
library(pscl)
zip_mod <- try(pscl::zeroinfl(formula_pois, data = DoctorVisits, dist = "poisson"), silent = TRUE)
if (class(zip_mod) != "try-error") summary(zip_mod)

# Diagnostics: residual plots and influence
par(mfrow = c(2,2))
plot(pois_mod)

# Pearson residuals check
pearson_resid <- residuals(pois_mod, type = "pearson")
plot(fitted(pois_mod), pearson_resid, ylab = "Pearson residuals", xlab = "Fitted values")
abline(h = 0, col = "red")

# Influence measures
infl <- influence.measures(pois_mod)
summary(infl)

#### Metodologia

* **Cel**: zbudować model przewidujący prawdopodobieństwo cukrzycy (`diabetes`), porównać linki: logit, probit, extreme-value (cloglog) oraz logistic-LASSO.
* **Wstęp**:

  * Używamy `PimaIndiansDiabetes2` z `mlbench`. Filtrujemy kompletne przypadki i ustawiamy `neg` jako referencję.
* **Modele parametryczne**:

  * `glm(..., family = binomial(link = "logit"))` oraz linki `probit` i `cloglog`.
  * Porównanie **AIC/BIC** i *apparent* AUC (do szybkiej oceny).
* **Logistic LASSO**:

  * `glmnet(..., family = "binomial")` z `cv.glmnet` (tutaj 10-fold CV dla AUC).
  * Wyciągamy ścieżkę i wybrane predyktory przy `lambda.min`.
* **Porównanie predykcyjne**:

  * `caret::train` z 10-fold CV i metryką ROC (AUC) dla każdej metody.
* **Kryterium wyboru**:

  * Zazwyczaj AUC (dla klasyfikacji) i stabilność zmiennych (LASSO). Równie istotna interpretowalność.
* **Uwagi**:

  * LASSO może usunąć ważne lecz skorelowane zmienne — warto sprawdzić stabilność wyboru poprzez bootstrapping.
  * Przy nierównych klasach (jeśli tak) rozważyć metody ważone lub oversampling.

In [ ]:
### ----------- Exercise 4: Diabetes (PimaIndiansDiabetes2) — Logistic / Probit / Extreme value + LASSO ----------- ###
# parameters
library(mlbench)
data("PimaIndiansDiabetes2", package = "mlbench")
pid2 <- PimaIndiansDiabetes2 %>% filter(complete.cases(.)) %>%
  mutate(diabetes = relevel(diabetes, ref = "neg"))

# prepare X and y
y_pid <- ifelse(pid2$diabetes == "pos", 1, 0)
X_pid <- pid2 %>% select(-diabetes)
Xmat_pid <- model.matrix(~ . -1, data = X_pid)

# 1) Logistic regression
logit_mod <- glm(diabetes ~ ., data = pid2, family = binomial(link = "logit"))
summary(logit_mod)

# 2) Probit
probit_mod <- glm(diabetes ~ ., data = pid2, family = binomial(link = "probit"))
summary(probit_mod)

# 3) Extreme value (cloglog link as extreme-value link)
cloglog_mod <- glm(diabetes ~ ., data = pid2, family = binomial(link = "cloglog"))
summary(cloglog_mod)

# Compare models using AIC / BIC and ROC AUC (10-fold CV AUC)
library(pROC)
predict_auc <- function(mod, data, response_name) {
  probs <- predict(mod, type = "response")
  roc_obj <- pROC::roc(data[[response_name]], probs, quiet = TRUE)
  auc(roc_obj)
}
cat("AICs:", AIC(logit_mod, probit_mod, cloglog_mod), "\n")
cat("BICs:", BIC(logit_mod, probit_mod, cloglog_mod), "\n")
cat("AUCs (apparent):",
    predict_auc(logit_mod, pid2, "diabetes"),
    predict_auc(probit_mod, pid2, "diabetes"),
    predict_auc(cloglog_mod, pid2, "diabetes"), "\n")

# 4) Logistic LASSO with glmnet (CV)
cv_glmnet_log <- cv.glmnet(Xmat_pid, y_pid, family = "binomial", alpha = 1, nfolds = 10, type.measure = "auc")
plot(cv_glmnet_log)
coef(cv_glmnet_log, s = "lambda.min")
# Selected predictors at lambda.min
selected_log_min <- rownames(coef(cv_glmnet_log))[which(as.numeric(coef(cv_glmnet_log, s = "lambda.min")) != 0)]
selected_log_min

# 5) Compare predictive performance via 10-fold CV (AUC) for selected methods (logit, probit, cloglog, lasso)
library(caret)
set.seed(2025)
ctrl <- trainControl(method = "cv", number = 10, classProbs = TRUE, summaryFunction = twoClassSummary)
# prepare data for caret (factor outcome with same levels)
pid2_caret <- pid2 %>% mutate(diabetes = factor(diabetes, levels = c("neg","pos")))
# logistic (caret)
train_log <- train(diabetes ~ ., data = pid2_caret, method = "glm", family = binomial, trControl = ctrl, metric = "ROC")
train_probit <- train(diabetes ~ ., data = pid2_caret, method = "glm", family = binomial(link = "probit"), trControl = ctrl, metric = "ROC")
train_clog <- train(diabetes ~ ., data = pid2_caret, method = "glm", family = binomial(link = "cloglog"), trControl = ctrl, metric = "ROC")
train_lasso_caret <- train(x = Xmat_pid, y = pid2_caret$diabetes, method = "glmnet", trControl = ctrl,
                           tuneGrid = expand.grid(alpha = 1, lambda = cv_glmnet_log$lambda), metric = "ROC")
resamps <- resamples(list(Logit = train_log, Probit = train_probit, CLogLog = train_clog, LASSO = train_lasso_caret))
summary(resamps)